# Database Operations (SQL-Focused)

This notebook provides database operations with:
- Connection management
- CRUD operations for scenarios
- ETF and benchmark data queries
- SQL-first approach (minimal pandas)

**Note:** This notebook uses the `database.py` module for all operations.

## Setup and Imports

In [ ]:
# Import database module
from database import db
from config import MYSQL_CONFIG
import pandas as pd

print("✅ Modules imported successfully")

## Connect to Database

In [ ]:
# Connect to MySQL database
if db.connect():
    print(f"✅ Connected to {MYSQL_CONFIG['database']}")
else:
    print("❌ Connection failed!")

## Demo 1: Get All ETFs

In [ ]:
# Get all ETFs from database
etfs = db.get_all_etfs()

if etfs:
    df = pd.DataFrame(etfs)
    print(f"📊 Total ETFs: {len(df)}\n")
    display(df.head(10))
    
    # Show distribution by asset class
    print("\n📈 Distribution by Asset Class:")
    print(df['asset_class'].value_counts())
else:
    print("❌ No ETFs found")

## Demo 2: Get Specific ETF

In [ ]:
# Get specific ETF by ticker
ticker = 'SPY'  # Change this to any ticker

etf = db.get_etf_by_ticker(ticker)

if etf:
    print(f"📊 ETF Information: {ticker}\n")
    for key, value in etf.items():
        print(f"   {key:20s}: {value}")
else:
    print(f"❌ ETF '{ticker}' not found")

## Demo 3: Get All Benchmark Portfolios

In [ ]:
# Get all benchmark portfolios
benchmarks = db.get_all_benchmarks()

if benchmarks:
    df = pd.DataFrame(benchmarks)
    print(f"📊 Total Benchmarks: {len(df)}\n")
    display(df[['benchmark_id', 'benchmark_name', 'risk_level', 'target_return']].head(15))
    
    print("\n📈 Distribution by Risk Level:")
    print(df['risk_level'].value_counts())
else:
    print("❌ No benchmarks found")

## Demo 4: Get Benchmark Holdings

In [ ]:
# Get holdings for a specific benchmark
benchmark_id = 1  # Change this to any benchmark_id from above

holdings = db.get_benchmark_holdings(benchmark_id)

if holdings:
    df = pd.DataFrame(holdings)
    print(f"📊 Holdings for: {df['benchmark_name'].iloc[0]}\n")
    display(df[['ticker_symbol', 'etf_name', 'target_weight']])
    
    print(f"\nTotal Weight: {df['target_weight'].sum():.2%}")
else:
    print(f"❌ No holdings found for benchmark {benchmark_id}")

## Demo 5: Create a New Scenario (CREATE)

In [ ]:
# Create a new backtest scenario
from datetime import datetime

scenario_data = {
    'benchmark_id': 1,  # Compare with Traditional 60/40
    'created_by': 'Student Team',
    'scenario_name': 'My First Test Portfolio',
    'initial_capital': 100000.00,
    'start_date': '2020-01-01',
    'end_date': '2024-12-31',
    'strategy_type': 'BUY_HOLD',
    'rebalance_freq': None,
    'monthly_contribution': 0
}

scenario_id = db.create_scenario(scenario_data)

if scenario_id:
    print(f"✅ Scenario created! ID: {scenario_id}")
    
    # Add some holdings
    holdings = [
        ('SPY', 0.60),  # 60% S&P 500
        ('AGG', 0.40)   # 40% Bonds
    ]
    
    for ticker, weight in holdings:
        etf = db.get_etf_by_ticker(ticker)
        if etf:
            db.add_scenario_holding(scenario_id, etf['etf_id'], weight)
            print(f"   Added {ticker}: {weight:.1%}")
    
    db.connection.commit()
    print(f"\n✅ Scenario {scenario_id} created with holdings!")
else:
    print("❌ Failed to create scenario")

## Demo 6: View All Scenarios (READ)

In [ ]:
# Get all scenarios
scenarios = db.get_all_scenarios()

if scenarios:
    df = pd.DataFrame(scenarios)
    print(f"📊 Total Scenarios: {len(df)}\n")
    display(df[['scenario_id', 'scenario_name', 'created_by', 'strategy_type', 'initial_capital']])
else:
    print("❌ No scenarios found. Create one first!")

## Demo 7: View Scenario Details (READ)

In [ ]:
# Get specific scenario details
scenario_id = 1  # Change to your scenario_id

scenario = db.get_scenario(scenario_id)

if scenario:
    print(f"📊 Scenario {scenario_id}: {scenario['scenario_name']}\n")
    
    for key, value in scenario.items():
        print(f"   {key:20s}: {value}")
    
    # Get holdings
    print("\n📈 Holdings:")
    holdings = db.get_scenario_holdings(scenario_id)
    if holdings:
        df = pd.DataFrame(holdings)
        display(df[['ticker_symbol', 'etf_name', 'target_weight']])
    else:
        print("   No holdings defined")
else:
    print(f"❌ Scenario {scenario_id} not found")

## Demo 8: Update Scenario (UPDATE)

In [ ]:
# Update an existing scenario
scenario_id = 1  # Change to your scenario_id

update_data = {
    'scenario_name': 'Updated Portfolio Name',
    'initial_capital': 150000.00,
    'start_date': '2020-01-01',
    'end_date': '2024-12-31',
    'strategy_type': 'REBALANCE',
    'rebalance_freq': 'QUARTERLY',
    'monthly_contribution': 1000
}

result = db.update_scenario(scenario_id, update_data)
if result is not None:
    db.connection.commit()
    print(f"✅ Scenario {scenario_id} updated successfully!")
    
    # Show updated data
    updated = db.get_scenario(scenario_id)
    if updated:
        print(f"\nUpdated values:")
        for key in update_data.keys():
            print(f"   {key:20s}: {updated[key]}")
else:
    print(f"❌ Failed to update scenario {scenario_id}")

## Demo 9: Delete Scenario (DELETE)

In [ ]:
# WARNING: This will delete the scenario and all related data!
# Uncomment to run

# scenario_id = 1  # Change to scenario you want to delete
# 
# result = db.delete_scenario(scenario_id)
# if result is not None:
#     db.connection.commit()
#     print(f"✅ Scenario {scenario_id} deleted successfully!")
# else:
#     print(f"❌ Failed to delete scenario {scenario_id}")

print("⚠️  Delete demo is commented out for safety")
print("    Uncomment the code above to test delete functionality")

## Demo 10: Get Price History

In [ ]:
# Get price history for an ETF
ticker = 'SPY'
etf = db.get_etf_by_ticker(ticker)

if etf:
    prices = db.get_price_history(
        etf['etf_id'],
        start_date='2024-01-01',
        end_date='2024-12-31'
    )
    
    if prices:
        df = pd.DataFrame(prices)
        print(f"📊 Price History for {ticker}\n")
        print(f"Total records: {len(df)}")
        print(f"Date range: {df['date'].min()} to {df['date'].max()}\n")
        display(df.head(10))
        
        # Quick stats
        print(f"\n📈 Quick Stats:")
        print(f"   Highest: ${df['high'].max():.2f}")
        print(f"   Lowest: ${df['low'].min():.2f}")
        print(f"   Latest Close: ${df['close'].iloc[-1]:.2f}")
    else:
        print(f"❌ No price history found for {ticker}")
else:
    print(f"❌ ETF {ticker} not found")

## Close Connection

In [ ]:
# Uncomment to close database connection
# db.disconnect()

print("ℹ️  Connection remains open. Uncomment above to close.")